In [1]:
import boto3
import json

In [2]:
bedrock = boto3.client(service_name = "bedrock-runtime" ,region_name = "us-east-1")
lambda_client = boto3.client(service_name = "lambda", region_name = "us-east-1")
MODEL_ID = "amazon.nova-micro-v1:0"

In [3]:
math_tool = {
    "toolSpec":{
        "name": "calculateNumbers",
        "description": "Perform basic arithmetic operations",
        "inputSchema":{
            "json":{
                "type":"object",
                "properties":{
                    "operation":{"type":"string"},
                    "num1":{"type": "number"},
                    "num2":{"type": "number"}
                },
                "required":["operation" ,"num1" ,"num2"]
            }
        }
    }
}

In [7]:
def execute_calculation(input_data):
    response = lambda_client.invoke(
        FunctionName = "math-function",
        InvocationType = "RequestResponse",
        Payload = json.dumps(input_data)
    )
    response_payload = response["Payload"].read()
    calculation_result = json.loads(response_payload)
    response_body = calculation_result.get("body", "{}")
    return json.loads(response_body) if isinstance(response_body, str) else response_body

User_input = {
    "role" :"user",
    "content":[{"text":"Please subtract 60 from 100"}]
}
system_prompt = [
    {
        "text": """
    You are a virtual assistant capable of performing basic arithmetic operations: add, subtract, multiply, and divide.
    If the user doesn't specify an operation, ask them for more details.
    """
    }
]
first_interaction = bedrock.converse(
    modelId = MODEL_ID,
    system = system_prompt,
    messages = [User_input],
    toolConfig = {
        "tools":[math_tool],
        "toolChoice":{"auto":{}}
    },
    inferenceConfig ={
        "temperature": 0.7
    }
)
assistant_reply = first_interaction["output"]["message"]
message_parts = assistant_reply["content"]
tool_request_block = next((part for part in message_parts if "toolUse" in part), None)
if not tool_request_block:
    print("=== Assistant's Direct Response ===")
    print(message_parts[0]["text"])
else:
    tool_request = tool_request_block["toolUse"]
    tool_input_data = tool_request["input"]
    tool_id = tool_request["toolUseId"]
    print(tool_request_block)
    print(f"→ Assistant triggered tool: calculateNumbers with input: {tool_input_data}")

tool_result = execute_calculation(tool_input_data)
print(f"← Lambda Function output: {tool_result}")
# Create a response based on the tool's output
try:
    result_summary = f"The outcome of the calculation is {tool_result['result']}."
except Exception as e:
    result_summary = f"Oops! There was an error with the calculation. ({str(e)})"

# Generate tool result message
tool_response_msg = {
    "role": "user",
    "content": [
        {
            "toolResult": {
                "toolUseId": tool_id,
                "content": [{"text": result_summary}]
            }
        }
    ]
}

# Send tool result back to the model
final_output = bedrock.converse(
    modelId=MODEL_ID,
    messages=[User_input, assistant_reply, tool_response_msg],
    toolConfig={  
        "tools": [math_tool],
        "toolChoice": {"auto": {}}
    },
    inferenceConfig={"temperature": 0.7}
)

# Display the final response from the assistant
final_message = final_output["output"]["message"]["content"][0]["text"]
print("\n=== Final Assistant Response ===")
print(final_message)

{'toolUse': {'toolUseId': 'tooluse_oTACi9zk4Ioyt55RYtSf5K', 'name': 'calculateNumbers', 'input': {'num1': 100, 'operation': 'subtract', 'num2': 60}}}
→ Assistant triggered tool: calculateNumbers with input: {'num1': 100, 'operation': 'subtract', 'num2': 60}
← Lambda Function output: {'result': 40.0}

=== Final Assistant Response ===
The outcome of subtracting 60 from 100 is 40.0.
